# Jamaica connectivity mapping based on condition

### Step 1: Imports and set up base paths and output path

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from rasterio.warp import reproject
from rasterio.enums import Resampling
import connectivity
import tifffile
import rioxarray as rxr
import os
from osgeo import gdal
import rioxarray
import Robynlibrary as Robyn
import Robyn_forest_classes

#### Define base paths and set inputs

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
processed_dir = base_path / "Processed_data"
output_dir = base_path / "Outputs"

land_use_path = base_path / "Inputs/2013_landuse_LandCover.shp"
hydrobasins_path = processed_dir / "HydroBASINS_Level12_Clipped_Jamaica.shp"


#### Read & reproject vector data

In [ ]:
terrestrial_landcover = gpd.read_file(land_use_path).copy() # Optional: if you want to preserve the original
jamaica_metric_grid_crs = "EPSG:3448"
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Reprojected CRS (terrestrial_landcover:", terrestrial_landcover.crs)

In [ ]:
hydrobasins = gpd.read_file(hydrobasins_path).copy()
print("Hydrobasins CRS:", hydrobasins.crs)

### Step 2: Read in condition spreadsheets and merge them with Jamaica landcover file 

In [ ]:
# Read your condition mapping spreadsheet
mapping_table = pd.read_excel(base_path / "landcover_condition_mapping.xlsx")
test_condition_mapping_ones = pd.read_excel(base_path / "jamaica_landcover_condition_test_ones.xlsx")
test_condition_mapping_zeros = pd.read_excel(base_path / "jamaica_landcover_condition_test_zeros.xlsx")
afforestable_condition_codes = pd.read_excel(base_path / "jamaica_afforestable_condition.xlsx")

In [ ]:
# Merge the mapping with your geodataframe on the 'Classify' column
landcover_with_condition_baseline = terrestrial_landcover.merge(mapping_table, on='Classify')
landcover_with_condition_ones = terrestrial_landcover.merge(test_condition_mapping_ones, on='Classify')
landcover_with_condition_zeros = terrestrial_landcover.merge(test_condition_mapping_zeros, on='Classify')
landcover_with_condition_afforestable = terrestrial_landcover.merge(afforestable_condition_codes, on='Classify')

### Step 3: Convert land use condition files to rasters

#### Define output files paths for the condition rasters 

In [ ]:
# Use the function for each condition and capture the outputs
baseline_out = output_dir / "landcover_condition.tif"
ones_out = output_dir / "landcover_condition_test_ones.tif"
zeros_out = output_dir / "landcover_condition_test_zeros.tif"
afforestable_out = output_dir / "landcover_condition_afforestable.tif"


#### Rasterize using the Robyn.rasterize_condition function 

In [ ]:
condition_raster_baseline, transform = Robyn.rasterize_condition(landcover_with_condition_baseline, baseline_out)
condition_raster_ones, _ = Robyn.rasterize_condition(landcover_with_condition_ones, ones_out)
condition_raster_zeros, _ = Robyn.rasterize_condition(landcover_with_condition_zeros, zeros_out)
condition_raster_afforestable, _ = Robyn.rasterize_condition(landcover_with_condition_afforestable, afforestable_out)

### Step 4: resample to 100 by 100 size cells rather than the current 10 by 10 

In [ ]:
# Compute the bounds from one of your GeoDataFrames (they should have the same extent)
bounds = landcover_with_condition_baseline.total_bounds

# Define output file names for the resampled rasters
baseline_resampled_file = output_dir / "landcover_condition_resampled.tif"
ones_resampled_file = output_dir / "landcover_condition_test_ones_resampled.tif"
zeros_resampled_file = output_dir / "landcover_condition_test_zeros_resampled.tif"
afforestable_resampled_file = output_dir / "landcover_condition_afforestable_resampled.tif"

# Apply the resampling function to each raster
Robyn.resample_and_save(condition_raster_baseline, transform, landcover_with_condition_baseline.crs, bounds, baseline_resampled_file)
Robyn.resample_and_save(condition_raster_ones, transform, landcover_with_condition_ones.crs, bounds, ones_resampled_file)
Robyn.resample_and_save(condition_raster_zeros, transform, landcover_with_condition_zeros.crs, bounds, zeros_resampled_file)
Robyn.resample_and_save(condition_raster_afforestable, transform, landcover_with_condition_afforestable.crs, bounds, afforestable_resampled_file)

### Step 5: Connectivity analysis 

In [ ]:
# Compute connectivity for each condition
baseline_connectivity = Robyn.compute_connectivity(baseline_resampled_file)
ones_connectivity = Robyn.compute_connectivity(ones_resampled_file)
zeros_connectivity = Robyn.compute_connectivity(zeros_resampled_file)
afforestable_connectivity = Robyn.compute_connectivity(afforestable_resampled_file)

print("Baseline connectivity:", baseline_connectivity)
print("Test ones connectivity:", ones_connectivity)
print("Test zeros connectivity:", zeros_connectivity)
print("Test afforestable connectivity:", afforestable_connectivity)

In [ ]:
# Define paths for the three resampled rasters
baseline_resampled_file = output_dir / "landcover_condition_resampled.tif"
ones_resampled_file = output_dir / "landcover_condition_test_ones_resampled.tif"
zeros_resampled_file = output_dir / "landcover_condition_test_zeros_resampled.tif"
afforestable_resampled_file = output_dir / "landcover_condition_afforestable_resampled.tif"

In [ ]:
# Example values
baseline_connectivity = 6257.130541768792
test_ones_connectivity = 9085.478043383626
test_zeros_connectivity = 526.1779088856231
test_afforestable_connectivity = 7158.552032105027

baseline_percentage = Robyn.calc_connectivity_percentage(baseline_connectivity, test_ones_connectivity, test_zeros_connectivity)
afforestable_percentage = Robyn.calc_connectivity_percentage(afforestable_connectivity, test_ones_connectivity, test_zeros_connectivity)

print(f"Baseline connectivity percentage: {baseline_percentage:.2f}%")
print(f"Afforestable connectivity percentage: {afforestable_percentage:.2f}%")

# Calculate the improvement (difference in percentage points)
improvement = afforestable_percentage - baseline_percentage
print(f"Improvement in connectivity percentage from baseline to afforestable: {improvement:.2f}%")

relative_improvement = ((afforestable_percentage - baseline_percentage) / baseline_percentage) * 100
print(f"Relative improvement compared to baseline: {relative_improvement:.2f}%")


### Step 6: calculate forest area as a proportion of Jamaica land cover area 

In [ ]:
# --- Compute total land area of Jamaica ---
total_land_area = terrestrial_landcover.geometry.area.sum()  # in m²

# --- Compute forest areas for both scenarios ---
baseline_forest_area = Robyn.compute_forest_area(terrestrial_landcover, Robyn_forest_classes.forest_classes, mixed=Robyn_forest_classes.mixed_classes)
afforestable_forest_area = Robyn.compute_forest_area(
    terrestrial_landcover, 
    Robyn_forest_classes.forest_plus_afforestable_classes_including_agricultural_classes, 
    mixed=Robyn_forest_classes.mixed_classes
)

# --- Compute proportions of total land area ---
baseline_forest_percentage = (baseline_forest_area / total_land_area) * 100
afforestable_forest_percentage = (afforestable_forest_area / total_land_area) * 100

# Convert areas to km² for easier interpretation
total_land_area_km2 = total_land_area / 1e6
baseline_forest_area_km2 = baseline_forest_area / 1e6
afforestable_forest_area_km2 = afforestable_forest_area / 1e6

# --- Print the results ---
print("Total land area of Jamaica: {:.2f} m² ({:.2f} km²)".format(total_land_area, total_land_area_km2))
print("Baseline forest area: {:.2f} m² ({:.2f} km²)".format(baseline_forest_area, baseline_forest_area_km2))
print("Baseline forest covers {:.2f}% of Jamaica".format(baseline_forest_percentage))
print("Afforestable forest area: {:.2f} m² ({:.2f} km²)".format(afforestable_forest_area, afforestable_forest_area_km2))
print("Afforestable forest covers {:.2f}% of Jamaica".format(afforestable_forest_percentage))

### Step 7: Catchment-level analysis 

In [ ]:
# Calculate the area in m² and km², and add them as new columns
hydrobasins["area_m2"] = hydrobasins.geometry.area
hydrobasins["area_km2"] = hydrobasins["area_m2"] / 1e6

# Add a new column with a unique new ID starting at 1
hydrobasins["new_id"] = range(1, len(hydrobasins) + 1)

# Calculate the equivalent diameter (distance across) in meters
# Equivalent diameter = 2 * sqrt(area_m2 / pi)
hydrobasins["equiv_diam_m"] = 2 * np.sqrt(hydrobasins["area_m2"] / np.pi)


# Calculate the smallest and largest HYBAS_ID area (in m² and km²)
smallest_area_m2 = hydrobasins["area_m2"].min()
mean_area_m2 = hydrobasins["area_m2"].mean()
largest_area_m2 = hydrobasins["area_m2"].max()
smallest_area_km2 = hydrobasins["area_km2"].min()
mean_area_km2 = hydrobasins["area_km2"].mean()
largest_area_km2 = hydrobasins["area_km2"].max()

smallest_diam = hydrobasins["equiv_diam_m"].min()
mean_diam = hydrobasins["equiv_diam_m"].mean()
largest_diam = hydrobasins["equiv_diam_m"].max()


# Print summary statistics of hydrobasins

print("Smallest HYBAS_ID area (m²):", smallest_area_m2)
print("Mean HYBAS_ID area (m²):", mean_area_m2)
print("Largest HYBAS_ID area (m²):", largest_area_m2)
print("Smallest HYBAS_ID area (km²):", smallest_area_km2)
print("Mean HYBAS_ID area (km²):", mean_area_km2)
print("Largest HYBAS_ID area (km²):", largest_area_km2)

print("Smallest equivalent diameter (m):", smallest_diam)
print("Mean equivalent diameter (m):", mean_diam)
print("Largest equivalent diameter (m):", largest_diam)
number_catchments = hydrobasins["new_id"].max()
print(number_catchments)

# Optionally, print a preview of the new columns along with the HYBAS_ID
print(hydrobasins[['HYBAS_ID', 'new_id', 'area_m2', 'area_km2', 'equiv_diam_m']].head())

# Write the modified hydrobasins to a new shapefile
hydrobasins.to_file("HydroBASINS_Level12_Clipped_Jamaica_modified.shp")

#### Rasterize hydrobasins 

In [ ]:
# Define the raster resolution (in meters) and extent
pixel_size = 100  # Change this value to your desired resolution (e.g., 10m)
minx, miny, maxx, maxy = hydrobasins.total_bounds

# Compute width and height in terms of pixels
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Create an affine transform for the raster (origin at top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Create (geometry, value) pairs for rasterization using the "new_id" column as the value.
shapes = ((geom, value) for geom, value in zip(hydrobasins.geometry, hydrobasins['new_id']))

# Rasterize the geometries into a NumPy array. 
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,  # Pixels that don't fall within any geometry will be assigned the "fill" value (0).
    dtype=np.uint16  # Change type as needed based on your data range
)

In [ ]:
# Define the output file path for saving the hydrobasins raster to a GeoTIFF file
hydrobasins_raster_path = processed_dir / "hydrobasins_raster.tif"
with rasterio.open(
    hydrobasins_raster_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,  # Number of bands; in this case one
    dtype=raster.dtype,
    crs=hydrobasins.crs.to_string(),
    transform=transform,
) as dst:
    dst.write(raster, 1)

print("Hydrobasins raster saved to:", hydrobasins_raster_path)

# Verify by opening the raster and printing its shape
with rasterio.open(str(hydrobasins_raster_path)) as src:
    resampled_array = src.read(1)  # Read the first band
print("Shape of resampled hydrobasins raster:", np.shape(resampled_array))

# Open the baseline landcover condition raster
baseline_raster_path = output_dir / "landcover_condition_resampled.tif"
with rasterio.open(str(baseline_raster_path)) as src:
    baseline_array = src.read(1)
print("Shape of baseline landcover raster:", np.shape(baseline_array))

# Use the baseline landcover raster as the template for reprojection
hydrobasins_raster_src = gdal.Open(str(hydrobasins_raster_path))
template_landcover = rxr.open_rasterio(str(baseline_raster_path)).squeeze()

# Reproject the source raster based on the template's resolution and extent
hydrobasins_raster_reproj = gdal.Warp(
    str(output_dir / "hydrobasins_raster_reproj.tif"),  # Output file name
    hydrobasins_raster_src,                              # Source dataset to be resampled
    xRes=template_landcover.rio.resolution()[0],         # X-resolution from the template
    yRes=template_landcover.rio.resolution()[0],         # Y-resolution from the template
    outputBounds=template_landcover.rio.bounds(),        # Output extent from the template
    dstSRS='EPSG:3448',                                  # Desired coordinate reference system
    resampleAlg="mode",
)

print("Reprojected raster saved to:", output_dir / "hydrobasins_raster_reproj.tif")

In [ ]:
# Get the GeoTransform and print bounds info from the reprojected hydrobasins raster
gt = hydrobasins_raster_reproj.GetGeoTransform()
print("Hydrobasins pixel width:", gt[1], "Pixel height:", gt[5])
print("Template resolution:", template_landcover.rio.resolution())
print("Template landcover bounds:", template_landcover.rio.bounds())

# Calculate the bounds of the reprojected raster using its geo-transform
minx = gt[0]
maxy = gt[3]
maxx = minx + (hydrobasins_raster_reproj.RasterXSize * gt[1])
miny = maxy - (hydrobasins_raster_reproj.RasterYSize * abs(gt[5]))

print("Reprojected hydrobasins bounds: ({}, {}, {}, {})".format(minx, miny, maxx, maxy))

#### Load rasters as arrays for further processing / validation 

In [ ]:
# Load and verify the values from the original and reprojected hydrobasins rasters.
original_hydrobasins_array = Robyn.open_raster_as_array(str(hydrobasins_raster_path))
reprojected_hydrobasins_array = Robyn.open_raster_as_array(str(output_dir / "hydrobasins_raster_reproj.tif"))

print("Unique hydrobasin IDs in original raster:", np.unique(original_hydrobasins_array))
print("Unique hydrobasin IDs in reprojected raster:", np.unique(reprojected_hydrobasins_array))

print("Shape of the reprojected hydrobasins raster:", reprojected_hydrobasins_array.shape)
print("Shape of the landcover bounds:", baseline_array.shape)


In [ ]:
# Read the baseline condition raster.
baseline_condition_raster = Robyn.open_raster_as_array(str(baseline_resampled_file))
print("Shape of catchment raster:", np.shape(reprojected_hydrobasins_array))
print("Shape of baseline condition raster:", np.shape(baseline_condition_raster))

# Determine the total number of catchments from the hydrobasins GeoDataFrame.
number_catchments = hydrobasins["new_id"].max()

# Loop over each catchment and calculate the non-zero pixel count.
for catchment_number in range(1, number_catchments + 1):
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(reprojected_hydrobasins_array == catchment_number)] = 1
    pixel_count = np.count_nonzero(catchment_temp)
    print(f"Catchment {catchment_number} - non-zero pixel count: {pixel_count}")

# Read the afforestable (afforested) condition raster.
afforested_condition_raster = Robyn.open_raster_as_array(str(afforestable_resampled_file))

# Setup parameters for connectivity analysis.
n_processes = os.cpu_count()
lambda_parameter = 5
generations_mode = "one_generation"
number_of_species_generations = 1

# Loop over each catchment to compute connectivity values.
for catchment_number in range(1, number_catchments + 1):
    # Create a binary mask for the current catchment.
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(reprojected_hydrobasins_array == catchment_number)] = 1
             
    # Compute connectivity for the current catchment under baseline conditions.
    baseline_connectivity_value = connectivity.landscape_connectivity(
        baseline_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )
    # Compute connectivity for the current catchment under the afforested scenario.
    afforested_connectivity_value = connectivity.landscape_connectivity(
        afforested_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )

    # Calculate the percentage difference in connectivity.
    perc_difference = (((afforested_connectivity_value - baseline_connectivity_value) / baseline_connectivity_value) * 100)
    print(f"Connectivity catchment {catchment_number}: baseline = {baseline_connectivity_value}, afforested = {afforested_connectivity_value}, difference = {perc_difference:.2f}%")
    
    # Update the hydrobasins GeoDataFrame with these connectivity metrics.
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "baseline_catchment_connectivity"] = baseline_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "afforested_catchment_connectivity"] = afforested_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"] == catchment_number, "percentage_difference_catchment_connectivity"] = perc_difference


In [ ]:
catchment_raster = open_raster_as_array(resampled_catchment_file)
print("Unique catchment IDs in raster:", np.unique(catchment_raster))
baseline_condition_raster = open_raster_as_array(baseline_resampled_file)

print(np.shape(catchment_raster))
print(np.shape(baseline_condition_raster))

for catchment_number in range(1, (number_catchments + 1)):
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(catchment_raster == catchment_number)] = 1
    pixel_count = np.count_nonzero(catchment_temp)
    print(f"Catchment {catchment_number} - non-zero pixel count: {pixel_count}")

afforested_condition_raster = open_raster_as_array(afforestable_resampled_file)

# Setup parameters for connectivity analysis
n_processes = os.cpu_count()
lambda_parameter = 5
generations_mode = "one_generation"
number_of_species_generations = 1

for catchment_number in range(1,(number_catchments+1)):
    catchment_temp = np.zeros_like(baseline_condition_raster)
    catchment_temp[np.where(catchment_raster == catchment_number)] = 1
             
    # Compute connectivity (using your connectivity module)
    baseline_connectivity_value = connectivity.landscape_connectivity(
        baseline_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )
    afforested_connectivity_value = connectivity.landscape_connectivity(
        afforested_condition_raster, 
        n_processes, 
        catchment_temp, 
        lambda_parameter, 
        generations_mode, 
        number_of_species_generations
    )

    perc_difference = (((afforested_connectivity_value - baseline_connectivity_value)/baseline_connectivity_value)*100) 
    print("connectivity catchment", catchment_number, "baseline: ", baseline_connectivity_value, "afforested: ", afforested_connectivity_value, "in %: ", perc_difference) 
    hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["baseline_catchment_connectivity"]] = baseline_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["afforested_catchment_connectivity"]] = afforested_connectivity_value
    hydrobasins.loc[hydrobasins["new_id"]==catchment_number, ["percentage_difference_catchment_connectivity"]] = perc_difference 





#### Condition scores in case of interest

In [ ]:
# # Baseline landcover condition mapping

 
# Disturbed broadleaved forest (Secondary Forest):	0.757	young secondary
# Fields: Herbaceous crops, fallow, cultivated v...:	0.684	annual cropland
# Secondary Forest:	0.855	mature secondary 
# Fields and Secondary Forest: 0.70225 75% annual cropland and 25% young secondary
# Buildings and other infrastructures: 0.717 urban
# Closed broadleaved forest (Primary Forest): 1 primary
# Plantation: Tree crops, shrub crops, sugar can...: 0.687 perennial crop
# Open dry forest - Tall (Woodland/Savanna): 1 primary
# Fields and Bamboo: 0.68475 - 25% annual cropland and 75% fields
# Bamboo and Secondary Forest: 0.729: 75% bamboo and 25% medium secondary
# Bamboo and Fields: 0.68625 - 75% and 25% annual cropland
# Herbaceous Wetland: 1 - primary
# Mangrove Forest: 1 - primary
# Fields or Secondary Forest/Pine Plantation: 0.635 - perennial crop
# Fields: Bare Land: 0.587 - managed pasture for now
# Fields: Pasture,Human disturbed, grassland: 0.587 - managed pasture for now
# Water Body: 0 
# Bamboo: 0.687: perennial for now but may want to make it look worse
# Bauxite Extraction: 0 
# Open dry forest - Short: 1 
# Bare Rock: 1
# Quarry: 0 
# Hardwood Plantation: Mahogany: 0.635 - perennial crop
# Swamp Forest: 1 - primary
# Hardwood Plantation: Euculytus: 0.635 - perennial crop
# Hardwood Plantation: Mixed: 0.635 - perennial crop
# Hardwood Plantation: Mahoe: 0.635 - perennial crop


# Afforestable landcover condition mapping
# Disturbed broadleaved forest (Secondary Forest):	0.757 - young secondary
# Fields: Herbaceous crops, fallow, cultivated v...:	0.757 - young secondary
# Secondary Forest:	0.855 - mature secondary
# Fields and Secondary Forest: 0.757 - young secondary
# Buildings and other infrastructures: 0.717 - urban
# Closed broadleaved forest (Primary Forest): 1 - primary
# Plantation: Tree crops, shrub crops, sugar can...: 0.757 - young secondary
# Open dry forest - Tall (Woodland/Savanna): 1 - primary
# Fields and Bamboo:  0.757 - young secondary
# Bamboo and Secondary Forest: 0.7815 - 75% young secondary and 25% medium secondary
# Bamboo and Fields: 0.757 - young secondary
# Herbaceous Wetland: 1 - primary
# Mangrove Forest: 1 - primary
# Fields or Secondary Forest/Pine Plantation: 0.757 - young secondary
# Fields: Bare Land: 0.757 - young secondary
# Fields: Pasture,Human disturbed, grassland: 0.757 - young secondary 
# Water Body: 0 for now
# Bamboo: 0.757 - young secondary
# Bauxite Extraction: 0.757 - young secondary
# Open dry forest - Short:  1 - primary
# Bare Rock: 1 for now
# Quarry:  0
# Hardwood Plantation: Mahogany: 0.757 - young secondary
# Swamp Forest: 1 - primary
# Hardwood Plantation: Euculytus: 0.757 - young secondary
# Hardwood Plantation: Mixed: 0.757 - young secondary
# Hardwood Plantation: Mahoe: 0.757 - young secondary







### *** OLD CODE (for applying to one file at a time) *** 

In [ ]:
# # Determine the bounds and resolution for the output raster
# minx, miny, maxx, maxy = landcover_with_condition_baseline.total_bounds
# resolution = 10  # Define an appropriate resolution (in the units of your CRS)
# width = int((maxx - minx) / resolution)
# height = int((maxy - miny) / resolution)
# transform = from_origin(minx, maxy, resolution, resolution)

In [ ]:
# # Prepare shapes for rasterization: tuple of (geometry, condition value)
# shapes = ((geom, value) for geom, value in zip(landcover_with_condition.geometry, landcover_with_condition['Condition']))


In [ ]:
# condition_raster = rasterize(
#     shapes=shapes,
#     out_shape=(height, width),
#     fill=0,  # Value for areas with no data
#     transform=transform,
#     dtype='float32'
# )

In [ ]:
# # Write the raster to a GeoTIFF
# with rasterio.open(
#     out_raster,
#     "w",
#     driver="GTiff",
#     height=height,
#     width=width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=transform,
# ) as dst:
#     dst.write(condition_raster, 1)

In [ ]:
# plt.figure(figsize=(10, 10))
# plt.imshow(landcover_with_condition_baseline, cmap='viridis')
# plt.colorbar(label='Condition Value')
# plt.title('Land Cover Condition Raster')
# plt.xlabel('Pixel Column')
# plt.ylabel('Pixel Row')
# plt.show()

In [ ]:
# # Save the 10 m raster
# out_raster = output_dir / "landcover_condition.tif"
# with rasterio.open(
#     out_raster,
#     "w",
#     driver="GTiff",
#     height=height,
#     width=width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=transform,
# ) as dst:
#     dst.write(condition_raster, 1)
# print(f"10 m resolution raster saved at: {out_raster}")

# #

In [ ]:
# def open_raster_as_array(path):
#     """
#     opens a tiff raster file as a numpy array
#     input:
#         path: path to the tiff raster (format: ...)
#     """
#     raster_path = path
#     raster = tifffile.imread(raster_path)
#     array = np.array(raster)
#     return array

In [ ]:
# # Set new resolution
# new_resolution = 100
# new_width = int((maxx - minx) / new_resolution)
# new_height = int((maxy - miny) / new_resolution)
# new_transform = from_origin(minx, maxy, new_resolution, new_resolution)



In [ ]:
# # Create an empty array for the resampled raster data
# resampled_raster = np.empty(shape=(new_height, new_width), dtype='float32')



In [ ]:
# # Resample using average method (you can change the resampling method if needed)
# reproject(
#     source=condition_raster,
#     destination=resampled_raster,
#     src_transform=transform,
#     src_crs=landcover_with_condition.crs,
#     dst_transform=new_transform,
#     dst_crs=landcover_with_condition.crs,
#     resampling=Resampling.average
# )


In [ ]:
# # Save the resampled (100 m) raster
# resampled_out_raster = output_dir / "landcover_condition_resampled.tif"
# with rasterio.open(
#     resampled_out_raster,
#     "w",
#     driver="GTiff",
#     height=new_height,
#     width=new_width,
#     count=1,
#     dtype='float32',
#     crs=landcover_with_condition.crs,
#     transform=new_transform,
# ) as dst:
#     dst.write(resampled_raster, 1)
# print(f"100 m resolution raster saved at: {resampled_out_raster}")

In [ ]:
# condition_layer = open_raster_as_array(output_dir / "landcover_condition_resampled.tif")

In [ ]:
# land_array = np.zeros_like(condition_layer)
# land_array[np.where(condition_layer != 0)] = 1

In [ ]:
# n_processes = os.cpu_count()

In [ ]:
# lambda_parameter = 5

In [ ]:
# generations_mode = "one_generation"
# number_of_species_generations = 1

In [ ]:
# connectivity_value = connectivity.landscape_connectivity(condition_layer, 
#                                                          n_processes, land_array, lambda_parameter, generations_mode, 
#                                                          number_of_species_generations)

# connectivity_value